# 01. 행동·관찰 마스크와 혼합 손실

## 목표
에이전트 궤적의 토큰 역할을 구분하고, 행동에는 RL 손실을, 관찰에는 세계 모델 cross-entropy를 적용하는 원리를 계산합니다. 실제 GRPO 전체 구현이 아니라 마스킹과 가중치의 직관에 집중합니다.

In [ ]:
from dataclasses import dataclass
from math import log

@dataclass(frozen=True)
class Token:
    text: str
    role: str  # prompt, action, observation
    probability: float

trajectory = [
    Token("문제를", "prompt", 0.90),
    Token("ls", "action", 0.60),
    Token("exit=0", "observation", 0.75),
    Token("cat", "action", 0.50),
    Token("README.md", "observation", 0.80),
]

In [ ]:
def mean_negative_log_likelihood(tokens, role):
    selected = [-log(token.probability) for token in tokens if token.role == role]
    return sum(selected) / len(selected)

advantage = 0.7
world_weight = 0.3
policy_loss = advantage * mean_negative_log_likelihood(trajectory, "action")
world_loss = mean_negative_log_likelihood(trajectory, "observation")
total_loss = policy_loss + world_weight * world_loss

print(f"정책 손실: {policy_loss:.4f}")
print(f"세계 모델 손실: {world_loss:.4f}")
print(f"혼합 손실: {total_loss:.4f}")
# 두 평균을 별도로 계산하는 이유는 관찰 토큰이 많다는 이유만으로
# 세계 모델 목표가 정책 목표를 압도하지 않게 하기 위해서입니다.

## 실험

1. `world_weight`를 0, 0.1, 1.0으로 바꿔 혼합 손실을 비교하세요.
2. 실패 궤적을 가정해 advantage를 0으로 바꾸세요. 정책 신호는 사라져도 관찰 학습은 남는지 확인하세요.
3. 관찰 확률을 낮춰 환경 예측이 어려울 때 손실이 어떻게 변하는지 살펴보세요.

> 실제 GRPO에는 중요도 비율, clipping, KL 규제와 그룹 advantage 계산이 더 필요합니다.